In [2]:
from sklearn import preprocessing
import numpy as np
import pandas as pd
import os


In [25]:
# 1. Load CSV
script_dir = os.getcwd()
file_path = os.path.join(script_dir, "german_credit.csv")  # Adjust filename as needed
df = pd.read_csv(file_path)
print(f"Loaded {os.path.basename(file_path)}")

# 2. Rename to match original German dataset format
rename_map = {
    "status": "Checking Account",
    "duration": "Duration",
    "credit_history": "Credit History",
    "purpose": "Purpose",
    "amount": "Credit Amount",
    "savings": "Savings",
    "employment_duration": "Employment",
    "installment_rate": "Installment Rate",
    "personal_status_sex": "Personal Status",
    "other_debtors": "Guarantors",
    "present_residence": "Residence Duration",
    "property": "Property",
    "age": "Age",
    "other_installment_plans": "Installment Plans",
    "housing": "Housing",
    "number_credits": "Existing Credits",
    "job": "Job",
    "people_liable": "Dependents",
    "telephone": "Telephone",
    "foreign_worker": "Foreign Worker",
    "credit_risk": "Credit Risk"
}
df.rename(columns=rename_map, inplace=True)

# 3. Target: convert "good"/"bad" to 0/1
df["Credit Risk"] = df["Credit Risk"].str.strip().str.lower().map({"good": 0, "bad": 1})
y = df["Credit Risk"].values

# 4. Ordinal encodings based on official codebook
ordered_categories = {
    "Checking Account": [
        "no checking account", "... < 0 DM", "0<= ... < 200 DM", "... >= 200 DM / salary for at least 1 year"
    ],
    "Savings": [
        "unknown/no savings account", "... < 100 DM", "100 <= ... <  500 DM",
        "500 <= ... < 1000 DM", "... >= 1000 DM"
    ],
    "Employment": [
        "unemployed", "< 1 yr", "1 <= ... < 4 yrs", "4 <= ... < 7 yrs", ">= 7 yrs"
    ]
}

# 5. Process features
X = df.drop(columns=["Credit Risk"])
feature_names = X.columns.tolist()
X_numpy = X.values.astype(object)

categorical_cols = [col for col in X.columns if X[col].dtype == "object"]
categorical_feature_options = {}

for col in categorical_cols:
    i = X.columns.get_loc(col)
    if col in ordered_categories:
        cat_order = ordered_categories[col]
        cat_map = {label: idx for idx, label in enumerate(cat_order)}
        X_numpy[:, i] = X[col].map(cat_map).astype(float)
        categorical_feature_options[i] = cat_order
    else:
        le = preprocessing.LabelEncoder()
        le.fit(X[col].dropna())
        X_numpy[:, i] = X[col].map(lambda v: le.transform([v])[0] if pd.notna(v) else np.nan)
        categorical_feature_options[i] = list(le.classes_)


Loaded german_credit.csv


In [26]:
df["Employment"].value_counts()

1 <= ... < 4 yrs    339
>= 7 yrs            253
4 <= ... < 7 yrs    174
< 1 yr              172
unemployed           62
Name: Employment, dtype: int64

In [27]:
# see how many nan in X_numpy per column
X_numpy

array([[0.0, 18, 0, ..., 0, 0, 0],
       [0.0, 9, 0, ..., 1, 0, 0],
       [1.0, 12, 4, ..., 0, 0, 0],
       ...,
       [3.0, 21, 0, ..., 0, 1, 0],
       [1.0, 12, 4, ..., 0, 1, 0],
       [0.0, 30, 4, ..., 0, 0, 0]], dtype=object)

In [6]:
label_distribution = df.groupby("Credit History")["Credit Risk"].value_counts(normalize=True).unstack()
# label_distribution.columns = ["Good (0)", "Bad (1)"]
print(label_distribution)


Credit Risk                                         0         1
Credit History                                                 
all credits at this bank paid back duly      0.829352  0.170648
critical account/other credits elsewhere     0.428571  0.571429
delay in paying off in the past              0.375000  0.625000
existing credits paid back duly till now     0.681818  0.318182
no credits taken/all credits paid back duly  0.681132  0.318868


In [29]:
# for all columns in categorical columns, print the possible labels
for col in categorical_cols:
    print(f"{col}: {X[col].unique()}")

Checking Account: ['no checking account' '... < 0 DM'
 '... >= 200 DM / salary for at least 1 year' '0<= ... < 200 DM']
Credit History: ['all credits at this bank paid back duly'
 'no credits taken/all credits paid back duly'
 'existing credits paid back duly till now'
 'delay in paying off in the past'
 'critical account/other credits elsewhere']
Purpose: ['car (used)' 'others' 'retraining' 'furniture/equipment' 'car (new)'
 'business' 'domestic appliances' 'radio/television' 'repairs' 'vacation']
Savings: ['unknown/no savings account' '... <  100 DM' '100 <= ... <  500 DM'
 '... >= 1000 DM' '500 <= ... < 1000 DM']
Employment: ['< 1 yr' '1 <= ... < 4 yrs' '4 <= ... < 7 yrs' 'unemployed' '>= 7 yrs']
Installment Rate: ['< 20' '25 <= ... < 35' '20 <= ... < 25' '>= 35']
Personal Status: ['female : non-single or male : single' 'male : married/widowed'
 'female : single' 'male : divorced/separated']
Guarantors: ['none' 'guarantor' 'co-applicant']
Residence Duration: ['>= 7 yrs' '1 <= ... < 

In [28]:
pd.crosstab(df["Credit History"], df["Credit Risk"], normalize="index")
pd.crosstab(df["Credit History"], df["Credit Risk"])


Credit Risk,0,1
Credit History,,
all credits at this bank paid back duly,243,50
critical account/other credits elsewhere,21,28
delay in paying off in the past,15,25
existing credits paid back duly till now,60,28
no credits taken/all credits paid back duly,361,169
